# Frank O'Hara — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the Frank O'Hara workshop group. Their chain used a **web search + cross-model critique loop** architecture — analyzing O'Hara's style and formal technique, questioning the model's defaults, then bouncing between models for critique and revision before generating the final poem in a fresh context.

```
                    ┌──── CONTEXT A (analysis) ─────────────┐
                    │                                       │
initial   ──▶  web search ──▶ style ──▶ formal poetic      │
generation      + analysis      │       technique           │
                                │           │               │
                                ▼           ▼               │
                          questioning                       │
                          (why did you                      │
                           do that?)                        │
                                │                           │
                    ┌───── CONTEXT B (critique) ────────┐   │
                    │           │                       │   │
                    │     critique ──▶ revision ─┐      │   │
                    │       ▲          (back to  │      │   │
                    │       └──────── A) ◀───────┘      │   │
                    └───────────────────────────────────┘   │
                                │                           │
                    ┌───── CONTEXT C (final) ───────────┐   │
                    │                                   │   │
                    │  new prompt ──▶ POEM ✦            │   │
                    │                                   │   │
                    └───────────────────────────────────┘   │
                    └───────────────────────────────────────┘
```

**What makes this chain interesting:** The group used *three different models* — Gemini for generation and analysis, Claude for critique, and ChatGPT for the final poem. Each model brought a different "perspective" to the same material. This notebook uses a single model (GPT-4o) but simulates the cross-model effect by using **separate API calls with no shared context** and different system prompts — so each step sees only what you explicitly pass it, not the full history.

A key discovery: the model defaulted to making O'Hara sound "jolly and cheery" (lots of exclamation marks), and when questioned about *why*, it revealed assumptions the group could correct.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: Initial Generation

The group started with a simple prompt and found it "did badly" — the model produced something that sounded generically upbeat and used too many exclamation marks. This is the baseline to improve on.

In [ ]:
# ── Step 1: Initial generation ───────────────────────────────────────────────

initial = ask(
    "Generate a poem in the style of Frank O'Hara."
)

print("INITIAL GENERATION")
print("═" * 60)
print(initial)
print("\n" + "═" * 60)
print("\n↓ The group found this 'did badly.' Look for the model's")
print("  defaults: probably too cheery, too many exclamation marks,")
print("  and a 'jolly' tone that misses O'Hara's actual register.")

---
## Step 2: Analyze Style + Formal Poetic Technique

The group's next move was analytical: they asked the model to search for and analyze existing O'Hara poems — separately for **style** (voice, tone, sensibility) and **formal poetic technique** (line breaks, syntax, structure). Splitting these into separate questions forces more detailed answers than asking for everything at once.

In [ ]:
# ── Step 2a: Style analysis ──────────────────────────────────────────────────

style = ask(
    """Analyze the STYLE of Frank O'Hara's poetry. Draw on specific poems
("The Day Lady Died," "Having a Coke with You," "A Step Away from Them,"
"Meditations in an Emergency," "Personal Poem," etc.).

Focus on:
- His tone and emotional register (it's NOT simply cheerful)
- How he handles intimacy — the casual and the devastating
- His relationship to New York City as setting and character
- The way he name-drops (people, streets, brands) and why
- His characteristic movement: the walk, the lunch hour, the aside
- What he does with time — how present tense and memory coexist

Be specific. Quote or reference actual lines where possible."""
)

print("STYLE ANALYSIS")
print("═" * 60)
print(style)

In [ ]:
# ── Step 2b: Formal poetic technique ─────────────────────────────────────────

technique = ask(
    """Analyze the FORMAL POETIC TECHNIQUE of Frank O'Hara's poetry.
Draw on specific poems.

Focus on:
- Line breaks — how he uses (or ignores) them
- Syntax — his long, run-on sentences vs. short declarations
- How he structures a poem (does he use stanzas? turns? closures?)
- Punctuation — what he uses and what he omits
- The "I do this, I do that" pattern and how it works as a
  structural device
- Enjambment, caesura, and the pace of reading
- How his poems END — what kind of closure (or non-closure)
  he favors

Be specific. This is about craft, not content."""
)

print("FORMAL TECHNIQUE ANALYSIS")
print("═" * 60)
print(technique)

---
## Step 3: Questioning — Why Did You Do That?

The group's sharpest move: instead of just asking for a better poem, they **questioned the model's choices** in the initial generation. They noticed the model made O'Hara sound "very jolly and cheery" with lots of exclamation marks, and asked *why*.

This surfaces the model's assumptions — its defaults become visible and correctable.

In [ ]:
# ── Step 3: Questioning ──────────────────────────────────────────────────────

questioning = ask(
    f"""You generated this poem in the style of Frank O'Hara:

{initial}

I have some questions about the choices you made:

1. Why does this poem sound so upbeat / cheerful? O'Hara's tone is
   more complex — casual on the surface but often melancholic,
   anxious, or elegiac underneath. What made you default to cheerful?

2. Why the exclamation marks? O'Hara uses them, but sparingly and
   with specific effect. What assumption led you to overuse them?

3. The poem reads like a *description* of an O'Hara poem rather than
   an O'Hara poem. What's the difference? Where did you describe
   instead of enact?

4. O'Hara's poems feel spontaneous but are actually carefully
   constructed. Does this poem feel spontaneous, or does it feel
   like it's performing spontaneity?

Be honest about your defaults and where they came from."""
)

print("QUESTIONING THE MODEL'S CHOICES")
print("═" * 60)
print(questioning)

---
## Step 4: Cross-Context Critique

The group originally switched to **Claude** for this step — a completely fresh model with no memory of the generation process. We simulate this with a clean API call: the critique sees *only* the poem and the analysis, not the conversation that produced them.

A fresh context is honest in a way that the generating context can't be — it sees the output, not the intention.

In [ ]:
# ── Step 4: Cross-context critique ───────────────────────────────────────────
# Fresh context — simulates switching to a different model.
# Only the poem and analysis are passed; no generation history.

critique = ask(
    f"""Here is a poem that attempts to imitate Frank O'Hara's style:

{initial}

Here is an analysis of O'Hara's actual style:

{style}

And here is an analysis of his formal technique:

{technique}

Critique the poem against these analyses. Be specific and direct:
1. Where does the poem match O'Hara's actual style?
2. Where does it fall short or feel like a parody?
3. Which formal techniques does it get right? Which does it miss?
4. Quote the weakest lines and explain why they fail.
5. What 3–5 specific changes would bring this closer to O'Hara?""",
    system="You are a poetry critic specializing in the New York School. Be direct and specific in your critique."
)

print("CROSS-CONTEXT CRITIQUE")
print("═" * 60)
print(critique)

---
## Step 5: Revision

Now we bring everything together — the style analysis, the formal technique analysis, the questioning of defaults, and the critique — and generate a revised poem. This goes back to the "original" context (as the group went back to Gemini for revision).

In [ ]:
# ── Step 5: Revision ─────────────────────────────────────────────────────────

revision = ask(
    f"""I need you to write a revised poem in the style of Frank O'Hara.

Here is an analysis of his style:
{style}

Here is an analysis of his formal technique:
{technique}

Here is a previous attempt and what was wrong with it:

PREVIOUS ATTEMPT:
{initial}

WHAT THE MODEL GOT WRONG (self-assessment):
{questioning}

CRITIQUE FROM A FRESH READER:
{critique}

Now write a new poem that addresses all of this. Specifically:
- Don't be generically cheerful — find O'Hara's real tone:
  casual, intimate, with sadness running underneath
- Use specific New York details (streets, names, places)
- Let the poem move like a walk or a lunch hour — one thing
  after another, with sudden depths
- Match his formal technique: run-on syntax, natural line breaks,
  minimal or purposeful punctuation
- Sound spontaneous without performing spontaneity

Write only the poem."""
)

print("REVISED POEM")
print("═" * 60)
print(revision)

---
## Step 6: Second Critique + Revision

The group looped between critique and revision — the diagram shows the loop arrow between these steps. Let's do one more round: critique the revision, then revise again.

In [ ]:
# ── Step 6a: Second critique ─────────────────────────────────────────────────

critique_2 = ask(
    f"""Here is a poem attempting to capture Frank O'Hara's voice:

{revision}

For reference, here is what O'Hara's style actually involves:
{style}

Critique this poem. It's a revision of an earlier draft — assume the
obvious problems (generic cheerfulness, exclamation marks) have been
addressed. Now look for subtler issues:

1. Does it actually sound like O'Hara, or like someone who has read
   about O'Hara?
2. Are the specific details genuinely specific, or "O'Hara-flavored"
   generics? (e.g., any old street name vs. a street that matters)
3. Does the poem have a real emotional undercurrent, or is it just
   listing things?
4. How does it end? O'Hara's endings are crucial — they often
   land on something devastating disguised as offhand.

Be specific. Quote lines.""",
    system="You are a poetry critic specializing in the New York School. Be direct and specific."
)

print("SECOND CRITIQUE")
print("═" * 60)
print(critique_2)

In [ ]:
# ── Step 6b: Second revision ─────────────────────────────────────────────────

revision_2 = ask(
    f"""Here is a poem in the style of Frank O'Hara:

{revision}

Here is a critique of it:

{critique_2}

Revise the poem to address this critique. Keep what works.
Fix what doesn't. Pay special attention to the ending —
it should land on something that feels both casual and devastating.

Write only the revised poem."""
)

print("SECOND REVISION")
print("═" * 60)
print(revision_2)

---
## Step 7: New Prompt → Final Poem

The group's final move: take everything learned and write a **completely new prompt** in a **fresh context** (they switched to ChatGPT). The fresh model hasn't seen the drafts or the revision history — it gets a single, refined prompt built from all the analysis.

This is powerful because the final model isn't anchored to any previous draft. It generates freely, but from a prompt that encodes everything the group learned.

In [ ]:
# ── Step 7: New prompt in fresh context ──────────────────────────────────────
# This simulates the group's final move: switching to ChatGPT with a
# new prompt. Fresh API call — no prior context.

# Build the new prompt from everything we've learned
NEW_PROMPT = f"""Write a poem in the voice of Frank O'Hara.

Here is what we know about his style:

STYLE:
{style}

FORMAL TECHNIQUE:
{technique}

COMMON MISTAKES TO AVOID (learned from previous attempts):
- Don't make the tone generically cheerful or "jolly" — O'Hara's
  casualness covers real feeling
- Don't overuse exclamation marks
- Don't describe an O'Hara poem — BE an O'Hara poem
- Don't perform spontaneity — find it
- The ending should be offhand and devastating at once

Write a poem that sounds like O'Hara on his lunch break in New York,
thinking about someone he loves. Let it move the way he moves:
one thing after another, specific and alive, with something underneath
that only surfaces in the last few lines.

Write only the poem. No title."""

final_poem = ask(NEW_PROMPT)

print("FINAL POEM (fresh context)")
print("═" * 60)
print(final_poem)

---
## Compare All Versions

Now look at the full progression. The initial generation was flat; the analysis built understanding; the critique loop refined it; and the fresh-context final generation used everything learned without being anchored to any draft.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. INITIAL GENERATION (no context)")
print("═" * 60)
print(initial)

print("\n" + "═" * 60)
print("2. FIRST REVISION (after analysis + critique)")
print("═" * 60)
print(revision)

print("\n" + "═" * 60)
print("3. SECOND REVISION (after second critique loop)")
print("═" * 60)
print(revision_2)

print("\n" + "═" * 60)
print("4. FINAL POEM (fresh context, new prompt)")
print("═" * 60)
print(final_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → What did separating style from formal technique reveal?")
print("  → What happened when you questioned the model's defaults?")
print("  → Did the critique loop improve the poem or just polish it?")
print("  → Is the fresh-context final poem better than the revised one?")
print("  → The group used 3 models. Does using 1 model with separate")
print("    contexts achieve a similar effect?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Feed in actual poems.** The group relied on the model's knowledge of O'Hara. Try pasting in 3–5 actual poems ("The Day Lady Died," "Having a Coke with You," "Personal Poem") and re-running the analysis. Does the style description change?

**Run more critique loops.** The group did multiple rounds between critique and revision. Try 3–4 rounds. Does the poem converge on something good, or start to lose energy?

**Question different defaults.** The group caught exclamation marks and cheerfulness. What other defaults does the model have for O'Hara? Try generating 3–4 poems and looking for patterns.

**Try the genuine cross-model approach.** If you have access to multiple models (Claude, Gemini, ChatGPT), try the group's original architecture: generate in one, critique in another, finalize in a third. Compare to the single-model version.

**Generate love song lyrics.** O'Hara's casual intimacy might translate beautifully to song lyrics — or the form might fight his style. Try it and find out.

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet